# Week 18 Optional B: Deep RAG Evaluation with DeepEval

## Going Beyond RAGAS: 6 Metrics for Compliance RAG

**Who is this for**: Data scientists who want a production evaluation suite they can
wire into the CI/CD pipeline in Weeks 19-20, and who work in a regulated domain where
missed policy references are compliance violations.

## The Problem with RAGAS-Only Evaluation

Week 18's main notebook uses RAGAS with `faithfulness` and `answer_relevancy`. These
two metrics tell you:
- Did the LLM hallucinate? (faithfulness)
- Did the LLM address the question? (answer_relevancy)

What they do NOT tell you:
- Did the RETRIEVER find ALL the relevant chunks? (contextual recall)
- Are the relevant chunks ranked HIGHER than irrelevant ones? (contextual precision)
- Is there overall retrieval noise? (contextual relevancy)
- Did the answer contradict any known fact? (hallucination)

In a compliance domain: a missed CTR rule is worse than an extra irrelevant chunk.
Contextual recall is the metric that catches "the retriever missed a critical policy."

## The 6 DeepEval Metrics

| Metric | What it asks | Compliance relevance |
|--------|--------------|----------------------|
| ContextualRecallMetric | Did retriever find ALL relevant docs? | CRITICAL - missed policy = violation |
| ContextualPrecisionMetric | Are relevant docs ranked higher? | Reranker quality in metric form |
| ContextualRelevancyMetric | Is retrieved context relevant overall? | Noise ratio measurement |
| FaithfulnessMetric | Is answer supported by context? | Hallucination vs policy text |
| AnswerRelevancyMetric | Does answer address the question? | Same as RAGAS answer_relevancy |
| HallucinationMetric | Does answer contradict known facts? | Factual accuracy beyond retrieval |

## CRITICAL: Custom Judge Required

DeepEval's native `BedrockModel` class uses Sonnet-class model IDs that the class
AWS account does NOT have enabled. You MUST wrap Claude Haiku 3 in a custom
`DeepEvalBaseLLM` subclass.

Section 0 implements this wrapper. Do not skip it.

## CI Integration

DeepEval metrics are Pytest-compatible. Running `deepeval test run` executes them in
a standard test suite. This is exactly how Week 19-20 wires evaluation into CI/CD.
The final lab in this notebook generates a `test_fraud_rag.py` file you carry into
Week 19.

## Prerequisites

- Week 18 main notebook completed
- `deepeval` installed (new dependency, see Section 0)

# Section 0: Setup + Custom Haiku Judge

## Why We Need a Custom Judge

DeepEval uses an LLM to evaluate your RAG system. By default it uses OpenAI. For
Bedrock, it provides a `BedrockModel` class - but that class targets Sonnet-class
model IDs by default. The class AWS account only has Haiku 3 enabled.

The fix is to subclass `DeepEvalBaseLLM` and implement `generate()` and `a_generate()`
methods that call `bedrock_runtime.converse()` with the Haiku 3 model ID.

This is the documented approach at deepeval.com/guides/guides-using-custom-llms.

## Pre-flight Probe

Always probe the custom judge before running any metrics. If it throws, find out here
rather than 10 minutes into a metric run.

In [ ]:
# Install required libraries. Versions are lower-bound only so pip resolves
# quickly from cache. sagemaker is pinned to v2.x because v3 removed
# get_execution_role() from the top-level namespace.
#
# matplotlib>=3.7: deepeval has a transitive matplotlib dep. Without an explicit
# pin, pip may attempt to build an old matplotlib from source on Python 3.12,
# which fails because configparser.SafeConfigParser was removed in 3.12.
# Pinning >=3.7 ensures pip resolves a pre-built wheel.

%pip install -q \
    "matplotlib>=3.7" \
    "sagemaker>=2.200,<3" \
    "deepeval" \
    "boto3>=1.35" \
    "strands-agents>=1.37" \
    "strands-agents-tools[mem0-memory]>=0.2.10" \
    "langchain>=0.3" \
    "langchain-aws>=0.2" \
    "langchain-community>=0.3" \
    "faiss-cpu>=1.9"

print("\nPackages installed. Restart kernel if first install.")

In [ ]:
# =============================================================================
# IMPORTS + SETUP
# =============================================================================
import os
import json
import asyncio
import boto3
import sagemaker
import pandas as pd
from sagemaker import get_execution_role
from importlib.metadata import version as pkg_version

# DeepEval imports
from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.test_case import LLMTestCase
from deepeval.dataset import EvaluationDataset
from deepeval import evaluate as deepeval_evaluate
from deepeval.metrics import (
    ContextualRecallMetric,
    ContextualPrecisionMetric,
    ContextualRelevancyMetric,
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    HallucinationMetric,
)

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_aws import BedrockEmbeddings
from langchain_core.documents import Document

try:
    print(f"  deepeval      {pkg_version('deepeval')}")
    print(f"  boto3         {pkg_version('boto3')}")
    print(f"  faiss-cpu     {pkg_version('faiss-cpu')}")
    print(f"  numpy         {pkg_version('numpy')}")
except Exception as e:
    print(f"  version check error: {e}")


# =============================================================================
# SAGEMAKER + BEDROCK SETUP
# =============================================================================
sess   = sagemaker.Session()
role   = get_execution_role()
AWS_REGION = sess.boto_region_name

os.environ["AWS_REGION"]         = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION
os.environ["KNOWLEDGE_BASE_ID"]  = (
    os.environ.get("KNOWLEDGE_BASE_ID")
    or os.environ.get("STRANDS_KNOWLEDGE_BASE_ID")
    or "FARSQGTONR"
)
STRANDS_KNOWLEDGE_BASE_ID = os.environ["KNOWLEDGE_BASE_ID"]

MODEL_ID       = "us.anthropic.claude-3-haiku-20240307-v1:0"
EMBED_MODEL_ID = "amazon.titan-embed-text-v2:0"

bedrock_runtime       = boto3.client("bedrock-runtime",       region_name=AWS_REGION)
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=AWS_REGION)
bedrock_embeddings    = BedrockEmbeddings(model_id=EMBED_MODEL_ID, region_name=AWS_REGION)

print(f"\nAWS Region: {AWS_REGION}")
print(f"LLM:        {MODEL_ID}")
print(f"KB:         {STRANDS_KNOWLEDGE_BASE_ID}")

In [ ]:
# =============================================================================
# CUSTOM DEEPEVAL JUDGE - Claude Haiku 3 via Bedrock Converse
# =============================================================================
# DeepEval's native BedrockModel targets Sonnet-class IDs. The class account
# only has Haiku 3 enabled. We MUST use DeepEvalBaseLLM with a custom wrapper.
#
# Pattern from: https://deepeval.com/guides/guides-using-custom-llms

class HaikuJudge(DeepEvalBaseLLM):
    """Custom DeepEval judge backed by Claude Haiku 3 via Bedrock Converse."""

    def load_model(self):
        # Return the boto3 client - DeepEvalBaseLLM calls this before generate()
        return bedrock_runtime

    def generate(self, prompt: str) -> str:
        # Synchronous generation - used by most metrics
        client = self.load_model()
        resp   = client.converse(
            modelId=MODEL_ID,
            messages=[{"role": "user", "content": [{"text": prompt}]}],
            inferenceConfig={"maxTokens": 1024, "temperature": 0},
        )
        return resp["output"]["message"]["content"][0]["text"]

    async def a_generate(self, prompt: str) -> str:
        # Async wrapper - runs the synchronous generate in a thread pool.
        # Do NOT use asyncio.run() here - it breaks in a running event loop (Jupyter).
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(None, self.generate, prompt)

    def get_model_name(self) -> str:
        return MODEL_ID


# =============================================================================
# PRE-FLIGHT: probe the custom judge before running any metrics
# =============================================================================
haiku_judge = HaikuJudge()
try:
    test_out = haiku_judge.generate("Respond with the single word: pong")
    print(f"Custom judge probe OK: {test_out!r}")
except Exception as e:
    print(f"Custom judge probe FAILED: {e}")
    print(f"Ask your instructor to enable Bedrock access for {MODEL_ID}.")
    raise

In [ ]:
# =============================================================================
# FRAUD CORPUS + LOCAL FAISS INDEX + RETRIEVAL HELPERS
# =============================================================================
# Same 8-document corpus as Week 18 main notebook.
# We rebuild a local FAISS index here so this notebook is self-contained.

FRAUD_POLICY_CORPUS = [
    ("ctr_rules.md",
     "Currency Transaction Report (CTR) rules under 31 CFR 1010.311. Banks "
     "must file a CTR for each transaction in currency of more than $10,000. "
     "Multiple transactions are aggregated when known to be conducted by or "
     "on behalf of the same person and result in cash in or cash out totaling "
     "more than $10,000 in any one business day. Structuring - breaking a "
     "transaction into smaller amounts to evade the CTR threshold - is itself "
     "a federal violation under 31 USC 5324."),
    ("ofac_screening.md",
     "OFAC screening requirements. All wire transfers must be screened against "
     "the OFAC Specially Designated Nationals (SDN) list before execution. "
     "International wire transfers involving countries on the OFAC sanctions "
     "list (including but not limited to Iran, North Korea, Syria, and Cuba) "
     "require additional review and may be blocked outright. False positives "
     "must be cleared within 24 hours."),
    ("wire_record_keeping.md",
     "Wire transfer recordkeeping under 31 CFR 1010.410. For any international "
     "wire transfer of $3,000 or more, the bank must retain the originator's "
     "name, address, account number, amount, execution date, payment "
     "instructions, beneficiary bank, and beneficiary name. Records must be "
     "retained for five years."),
    ("structuring_red_flags.md",
     "Structuring red flags. Multiple cash deposits of amounts just under "
     "$10,000 across consecutive days at the same or related accounts are a "
     "classic structuring pattern. Velocity anomalies - for example three or "
     "more transactions at unrelated merchants within 30 minutes - indicate "
     "potential card testing or account takeover."),
    ("account_takeover_patterns.md",
     "Account takeover (ATO) indicators. A password change followed within "
     "minutes by a wire transfer to a newly added payee from an unfamiliar IP "
     "is a high-confidence ATO signal. Transactions that originate from "
     "geographies inconsistent with the customer's historical footprint, "
     "especially from countries the customer has never transacted with, "
     "require hold and verification."),
    ("unusual_hours_rule.md",
     "Unusual hours rule. Transactions initiated between 1:00 AM and 5:00 AM "
     "local time that fall outside the customer's typical active window "
     "require enhanced monitoring. Two or more such transactions within a "
     "single session should trigger a SAR review."),
    ("new_payee_hold.md",
     "New payee large transfer hold. Wire transfers exceeding $5,000 to payees "
     "that were added to the account within the preceding 72 hours require "
     "two-factor customer verification and a 24-hour hold regardless of the "
     "customer's risk score."),
    ("high_risk_merchant_categories.md",
     "High-risk merchant category codes (MCCs). Cryptocurrency exchanges, "
     "offshore gambling platforms, and money transfer services are classified "
     "as high-risk MCCs. Transactions at these merchants for amounts over "
     "$1,000 require enhanced due diligence."),
]

# Build FAISS index from local corpus (512-token chunks, 80-token overlap)
documents   = [Document(page_content=text, metadata={"source": name})
               for name, text in FRAUD_POLICY_CORPUS]
splitter    = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=80)
chunks      = splitter.split_documents(documents)
faiss_index = FAISS.from_documents(chunks, bedrock_embeddings)


def retrieve_contexts(question: str, k: int = 3) -> list:
    """Retrieve top-k contexts from Bedrock KB (raw text list for DeepEval).
    
    Uses the Bedrock KB (not the local FAISS) so results match production.
    Falls back to FAISS if the KB call fails.
    """
    try:
        resp = bedrock_agent_runtime.retrieve(
            knowledgeBaseId=STRANDS_KNOWLEDGE_BASE_ID,
            retrievalQuery={"text": question},
            retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": k}},
        )
        return [i.get("content", {}).get("text", "")
                for i in resp.get("retrievalResults", [])]
    except Exception:
        # Fallback to local FAISS during development
        docs = faiss_index.similarity_search(question, k=k)
        return [d.page_content for d in docs]


def generate_answer(question: str, contexts: list) -> str:
    """Generate a RAG answer from retrieved contexts using Claude Haiku."""
    context_text = "\n".join(contexts)
    prompt = (
        f"Answer using only the context below. Be specific and cite regulation codes "
        f"and dollar amounts where relevant.\n\nContext:\n{context_text}\n\n"
        f"Question: {question}\nAnswer:"
    )
    resp = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 512, "temperature": 0},
    )
    return resp["output"]["message"]["content"][0]["text"]


print(f"Corpus: {len(FRAUD_POLICY_CORPUS)} docs -> {len(chunks)} chunks")
print("retrieve_contexts() and generate_answer() ready.")

# Section 1: The 6 DeepEval Metrics

## DeepEval vs RAGAS

Both frameworks use LLM-as-judge. The key difference is what they judge:

| | RAGAS v0.4 | DeepEval |
|--|-----------|---------|
| Faithfulness | Yes | Yes (FaithfulnessMetric) |
| Answer relevancy | Yes | Yes (AnswerRelevancyMetric) |
| Context precision | Yes | Yes (ContextualPrecisionMetric) |
| Context recall | Needs ground truth | Yes, REQUIRES ground truth |
| Hallucination | No | Yes (HallucinationMetric) |
| Pytest integration | No | Yes (deepeval test run) |

## LLMTestCase Structure

DeepEval uses `LLMTestCase` instead of RAGAS's `SingleTurnSample`:

```python
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input="What is the CTR threshold?",
    actual_output="Banks must file a CTR for cash transactions exceeding $10,000.",
    expected_output="More than $10,000 in a single business day triggers a CTR.",  # ground truth
    retrieval_context=["...chunk 1...", "...chunk 2..."],  # raw text list
    context=["...chunk 1...", "...chunk 2..."],            # same - required for hallucination metric
)
```

**Note on `expected_output`**: `ContextualRecallMetric` requires `expected_output`
(ground truth answer). Writing ground truth is intentional - it forces you to
formalize what a correct answer looks like. That IS the hard part of eval.

## The Threshold Parameter

Each metric takes a `threshold` parameter (0 to 1). A test case PASSES if the metric
score >= threshold. The default is 0.5; for compliance you likely want 0.7-0.8.

```python
metric = FaithfulnessMetric(threshold=0.7, model=haiku_judge)
```

**Exception**: `HallucinationMetric` is inverted - a lower score is better (less
hallucination). Set its threshold to 0.3 meaning "fail if hallucination score > 0.3".

In [ ]:
# =============================================================================
# DEMO: Run 3 DeepEval Metrics on One Test Case
# =============================================================================

DEMO_QUESTION = "What is the CTR reporting threshold for cash transactions?"

# Ground truth: what a compliance expert would say (we author this, not the LLM)
DEMO_GROUND_TRUTH = (
    "Banks must file a Currency Transaction Report (CTR) for each cash transaction "
    "exceeding $10,000. Multiple transactions in the same business day by or for the "
    "same person are aggregated and also require a CTR if the total exceeds $10,000. "
    "Structuring transactions to avoid the CTR threshold is a federal violation under "
    "31 USC 5324."
)

# Retrieve + generate the actual RAG answer
contexts = retrieve_contexts(DEMO_QUESTION, k=3)
answer   = generate_answer(DEMO_QUESTION, contexts)

print(f"Question: {DEMO_QUESTION}")
print(f"\nAnswer: {answer[:300]}...")
print(f"\nRetrieved {len(contexts)} context chunks")

# Assemble the test case (all fields required for recall + hallucination)
demo_case = LLMTestCase(
    input=DEMO_QUESTION,
    actual_output=answer,
    expected_output=DEMO_GROUND_TRUTH,
    retrieval_context=contexts,
    context=contexts,
)

# Run 3 metrics - one retriever metric, one generator metric, one recall metric
metrics = [
    FaithfulnessMetric(     threshold=0.7, model=haiku_judge),
    AnswerRelevancyMetric(  threshold=0.7, model=haiku_judge),
    ContextualRecallMetric( threshold=0.7, model=haiku_judge),
]

deepeval_evaluate(test_cases=[demo_case], metrics=metrics)

print("\nDEMO test case results:")
for m in metrics:
    status = "PASS" if m.score >= m.threshold else "FAIL"
    print(f"  {m.__class__.__name__:30s}  score={m.score:.3f}  [{status}]")
    if hasattr(m, "reason") and m.reason:
        print(f"    reason: {m.reason[:120]}")

## Lab 1: Author 5 Test Cases with Ground Truth (15 min)

### Your Task

Author 5 test cases for the fraud policy KB. This lab is intentionally about
AUTHORING - writing the ground truth is the hard part of RAG evaluation.

### Steps

1. Choose 5 fraud questions covering different topics: CTR, OFAC, ATO, structuring,
   MCC, or wire recordkeeping.
2. For each, write an `expected_output` string (the ideal answer a compliance expert
   would give). Be specific: include dollar amounts, regulation codes, timeframes.
   This is your ground truth - do not just copy from the corpus.
3. Build 5 `LLMTestCase` objects:
   - `input`: your question
   - `actual_output`: use `generate_answer(q, retrieve_contexts(q))`
   - `expected_output`: your authored ground truth
   - `retrieval_context` and `context`: from `retrieve_contexts(q, k=3)`
4. Run `FaithfulnessMetric` and `AnswerRelevancyMetric` on all 5 cases using
   `deepeval_evaluate(test_cases=lab1_cases, metrics=lab1_metrics)`.
5. Build a results DataFrame: question (truncated to 60 chars), faithfulness score,
   answer_relevancy score, faithfulness pass (bool), relevancy pass (bool).

### Expected Output

A 5-row DataFrame. Cases with faithfulness < 0.7 are worth investigating - the LLM
may be answering with information not supported by the retrieved chunks.

### Think About It

Writing the ground truth took time. How often would you re-run this eval? Who at your
organization would be responsible for maintaining ground truth as policies change?
(Hint: not the data scientist.)

In [ ]:
# =============================================================================
# SOLUTION: Lab 1 - 5 Test Cases with Ground Truth
# =============================================================================

LAB1_QA = [
    {
        "question":     "What is the CTR reporting threshold for cash transactions?",
        "ground_truth": (
            "Banks must file a CTR for cash transactions exceeding $10,000. "
            "Multiple same-day transactions by the same person are aggregated. "
            "Structuring to evade the threshold is a federal violation under 31 USC 5324."
        ),
    },
    {
        "question":     "Which countries require additional review for wire transfers under OFAC?",
        "ground_truth": (
            "Wire transfers to Iran, North Korea, Syria, and Cuba require additional "
            "OFAC review and may be blocked. False positives must be cleared within 24 hours."
        ),
    },
    {
        "question":     "What recordkeeping is required for international wire transfers of $3,000 or more?",
        "ground_truth": (
            "Under 31 CFR 1010.410, the bank must retain the originator's name, address, "
            "account number, amount, execution date, payment instructions, beneficiary bank, "
            "and beneficiary name. Records must be kept for five years."
        ),
    },
    {
        "question":     "What are the signs of a structuring pattern?",
        "ground_truth": (
            "Multiple cash deposits just under $10,000 over consecutive days are a classic "
            "structuring pattern. Velocity anomalies - three or more transactions at unrelated "
            "merchants within 30 minutes - also indicate potential card testing."
        ),
    },
    {
        "question":     "When must a bank place a hold on a wire transfer to a new payee?",
        "ground_truth": (
            "Wire transfers exceeding $5,000 to payees added within the preceding 72 hours "
            "require two-factor customer verification and a 24-hour hold regardless of the "
            "customer's risk score."
        ),
    },
]

# Build LLMTestCase objects: retrieve + generate for each question
lab1_cases = []
for qa in LAB1_QA:
    ctxts  = retrieve_contexts(qa["question"], k=3)
    answer = generate_answer(qa["question"], ctxts)
    lab1_cases.append(LLMTestCase(
        input=qa["question"],
        actual_output=answer,
        expected_output=qa["ground_truth"],
        retrieval_context=ctxts,
        context=ctxts,  # required by HallucinationMetric in later labs
    ))

lab1_metrics = [
    FaithfulnessMetric(    threshold=0.7, model=haiku_judge),
    AnswerRelevancyMetric( threshold=0.7, model=haiku_judge),
]

deepeval_evaluate(test_cases=lab1_cases, metrics=lab1_metrics)

# Build results DataFrame - re-measure per case to get individual scores
rows = []
for case in lab1_cases:
    m_faith = FaithfulnessMetric(    threshold=0.7, model=haiku_judge)
    m_rel   = AnswerRelevancyMetric( threshold=0.7, model=haiku_judge)
    m_faith.measure(case)
    m_rel.measure(case)
    rows.append({
        "question":         case.input[:60],
        "faithfulness":     round(m_faith.score, 3),
        "answer_relevancy": round(m_rel.score, 3),
        "faith_pass":       m_faith.score >= 0.7,
        "relevancy_pass":   m_rel.score >= 0.7,
    })

lab1_df = pd.DataFrame(rows)
print(lab1_df.to_string(index=False))

# Explanation:
# - We re-run .measure() per case to get per-case scores (deepeval_evaluate aggregates).
# - expected_output is required for ContextualRecallMetric in Lab 3; include it here.
# - Cases with faithfulness < 0.7 often indicate the LLM paraphrased beyond the context.

# Section 2: Context Recall and Why It Requires Ground Truth

## Context Recall: The Compliance-Critical Metric

`ContextualRecallMetric` asks: "Does the retrieval context contain ALL the information
needed to produce the ground truth answer?"

It works by checking whether each statement in the `expected_output` can be attributed
to the `retrieval_context`. If the retriever misses a chunk, the ground truth statement
it would have supported gets a zero score for that statement.

**Why this is the most important metric for compliance RAG**:

- RAGAS faithfulness checks if the answer is supported by WHAT WAS RETRIEVED
- Context recall checks if EVERYTHING RELEVANT was retrieved
- If your retriever misses the "structuring is a federal violation" chunk, but
  faithfulness is high (because the answer is consistent with what was retrieved),
  the system looks fine but has a regulatory gap

## The Recall-Precision Tension

| Metric | Goes up when you... | Goes down when you... |
|--------|---------------------|----------------------|
| Context recall | Retrieve more chunks (higher k) | Miss any relevant chunk |
| Context precision | Retrieve only relevant chunks | Return noise chunks |
| Retrieval relevancy | Retrieve focused results | Return off-topic chunks |

Increasing k improves recall but hurts precision. This is the fundamental tradeoff
your reranker from the Week 18 main notebook was designed to resolve.

## Demo: Recall at k=1 vs k=3

The next cell deliberately tests a case where the retriever needs both the CTR rule
AND the structuring penalty to fully answer the question. With k=1 it may miss one.

In [ ]:
# =============================================================================
# DEMO: ContextualRecallMetric - k=1 vs k=3
# =============================================================================
# Ground truth covers TWO facts: (1) the $10,000 CTR threshold and
# (2) structuring being a separate federal violation under 31 USC 5324.
# With k=1 the retriever may only find one of those facts.

Q  = "What is the CTR threshold and what happens if someone structures transactions to avoid it?"
GT = (
    "Banks must file a CTR for cash transactions exceeding $10,000 in a business day. "
    "Structuring transactions to evade the CTR threshold (breaking large transactions "
    "into smaller sub-$10,000 amounts) is itself a federal crime under 31 USC 5324, "
    "separate from the underlying CTR violation."
)

print("Comparing ContextualRecall at different k values:\n")
for k in [1, 3]:
    ctxts  = retrieve_contexts(Q, k=k)
    answer = generate_answer(Q, ctxts)

    case = LLMTestCase(
        input=Q,
        actual_output=answer,
        expected_output=GT,
        retrieval_context=ctxts,
        context=ctxts,
    )
    m = ContextualRecallMetric(threshold=0.7, model=haiku_judge)
    m.measure(case)
    status = "PASS" if m.score >= m.threshold else "FAIL"
    print(f"k={k}: recall={m.score:.3f}  [{status}]  chunks retrieved={len(ctxts)}")
    if m.reason:
        print(f"  reason: {m.reason[:200]}")

print(
    "\nObservation: recall improves with more chunks because both the CTR threshold and "
    "the structuring violation are covered. Recall is a RETRIEVER quality metric."
)

## Lab 2: Recall Sensitivity Analysis (15 min)

### Your Task

Measure how context recall changes as you vary the number of retrieved chunks (k).
This directly informs the k value you should use in production.

### Steps

1. Pick 3 questions from Lab 1 that have multi-part ground truths - answers that need
   more than one policy document to be complete.
2. For each question, run `ContextualRecallMetric` with k = 1, 3, 5, 8.
   Use `threshold=0.5` so you see the gradient, not just pass/fail.
3. Collect results into a DataFrame with columns: question (truncated), k, recall_score.
4. Identify the "knee" - the k value after which recall stops improving.

### Expected Output

A 12-row DataFrame (3 questions x 4 k values) with recall scores. You should see
diminishing returns: recall improves from k=1 to k=3, then flattens from k=5 to k=8.

### Think About It

What is the cost of increasing k in production? (More tokens sent to the LLM generator,
higher latency, higher API cost.) At what k does the compliance benefit outweigh the cost?
Who in the organization decides that tradeoff?

In [ ]:
# =============================================================================
# SOLUTION: Lab 2 - Recall Sensitivity Analysis
# =============================================================================
# Pick 3 multi-part questions from Lab 1 and sweep k = 1, 3, 5, 8.

lab2_questions_gt = [
    (
        "What is the CTR threshold and what happens if someone structures transactions to avoid it?",
        "Banks must file a CTR for cash transactions exceeding $10,000. Structuring to evade "
        "the threshold is a federal crime under 31 USC 5324, separate from the CTR violation.",
    ),
    (
        "What recordkeeping is required for international wire transfers of $3,000 or more?",
        "Under 31 CFR 1010.410, retain originator name, address, account number, amount, "
        "execution date, payment instructions, beneficiary bank, and beneficiary name for five years.",
    ),
    (
        "Which countries require additional review for wire transfers under OFAC?",
        "Wire transfers to Iran, North Korea, Syria, and Cuba require OFAC review and may be "
        "blocked outright. False positives must be cleared within 24 hours.",
    ),
]

rows = []
for q, gt in lab2_questions_gt:
    for k in [1, 3, 5, 8]:
        ctxts  = retrieve_contexts(q, k=k)
        answer = generate_answer(q, ctxts)
        case   = LLMTestCase(
            input=q,
            actual_output=answer,
            expected_output=gt,
            retrieval_context=ctxts,
            context=ctxts,
        )
        m = ContextualRecallMetric(threshold=0.5, model=haiku_judge)
        m.measure(case)
        rows.append({
            "question":     q[:50],
            "k":            k,
            "recall_score": round(m.score, 3),
        })

lab2_df = pd.DataFrame(rows)
print(lab2_df.to_string(index=False))

# Explanation:
# - threshold=0.5 shows the gradient; strict 0.7 would show mostly FAIL at low k.
# - Expected pattern: recall improves k=1 -> k=3, flattens or plateaus k=5 -> k=8.
# - The "knee" is typically k=3 for this 8-doc corpus - adding more chunks brings
#   diminishing returns once the relevant docs are already retrieved.
# - The cost of k=8 vs k=3: roughly 2.6x more tokens sent to the generator per query.

# Section 3: The Full 6-Metric Suite

## Running All 6 Metrics Together

In production you run all 6 metrics on every evaluation dataset push. Any single
metric below threshold blocks the deployment.

The gate looks like this:

```
LLMTestCase --> ContextualRecall     --> PASS/FAIL
            --> ContextualPrecision  --> PASS/FAIL
            --> ContextualRelevancy  --> PASS/FAIL
            --> Faithfulness         --> PASS/FAIL
            --> AnswerRelevancy      --> PASS/FAIL
            --> Hallucination        --> PASS/FAIL
                                         |
                             ALL PASS? ---> Deploy
                             ANY FAIL? ---> Block + report which metric
```

## FaithfulnessMetric vs HallucinationMetric

These two are complementary, not redundant:

- **FaithfulnessMetric**: Is the answer SUPPORTED by the context? (positive check)
  Score = fraction of answer statements attributable to retrieval context.
  Higher is better. Threshold 0.7.

- **HallucinationMetric**: Does the answer CONTRADICT the context? (negative check)
  Score = degree of contradiction. Lower is better. Threshold 0.3.

A vague, non-committal answer can score high on faithfulness (nothing to contradict)
while also being useless. That is why both metrics run together.

## HallucinationMetric Threshold Direction

```python
# All other metrics: PASS if score >= threshold (higher is better)
FaithfulnessMetric(threshold=0.7, ...)   # score=0.8 -> PASS

# HallucinationMetric: PASS if score <= threshold (lower is better)
HallucinationMetric(threshold=0.3, ...)  # score=0.1 -> PASS (low hallucination)
                                          # score=0.8 -> FAIL (high hallucination)
```

In [ ]:
# =============================================================================
# DEMO: All 6 DeepEval Metrics on One Test Case
# =============================================================================

Q  = "What are the recordkeeping requirements for international wire transfers?"
GT = (
    "For any international wire transfer of $3,000 or more, the bank must retain "
    "the originator's name, address, account number, transaction amount, execution "
    "date, payment instructions, beneficiary bank, and beneficiary name. Records "
    "must be kept for five years. This is required under 31 CFR 1010.410."
)

ctxts  = retrieve_contexts(Q, k=4)
answer = generate_answer(Q, ctxts)

full_case = LLMTestCase(
    input=Q,
    actual_output=answer,
    expected_output=GT,
    retrieval_context=ctxts,
    context=ctxts,
)

# All 6 metrics instantiated with the custom Haiku judge
six_metrics = [
    ContextualRecallMetric(    threshold=0.7, model=haiku_judge),
    ContextualPrecisionMetric( threshold=0.7, model=haiku_judge),
    ContextualRelevancyMetric( threshold=0.7, model=haiku_judge),
    FaithfulnessMetric(        threshold=0.7, model=haiku_judge),
    AnswerRelevancyMetric(     threshold=0.7, model=haiku_judge),
    HallucinationMetric(       threshold=0.3, model=haiku_judge),  # lower = less hallucination
]

deepeval_evaluate(test_cases=[full_case], metrics=six_metrics)

print("\nFull 6-metric results:")
for m in six_metrics:
    # HallucinationMetric passes when score <= threshold (inverted logic)
    if isinstance(m, HallucinationMetric):
        status = "PASS" if m.score <= m.threshold else "FAIL"
    else:
        status = "PASS" if m.score >= m.threshold else "FAIL"
    print(f"  {m.__class__.__name__:28s}  score={m.score:.3f}  [{status}]")

## Lab 3: Full 6-Metric Suite on 5 Questions (15 min)

### Your Task

Run the full 6-metric suite on your 5 test cases from Lab 1.

### Steps

1. Reuse `lab1_cases` (the 5 `LLMTestCase` objects from Lab 1, or the safety-net).
2. Instantiate all 6 metrics with `model=haiku_judge`.
3. Run `deepeval_evaluate(test_cases=lab1_cases, metrics=six_metrics_lab3)`.
4. Build a results DataFrame with columns:
   - `question` (truncated to 60 chars)
   - `contextual_recall`, `contextual_precision`, `contextual_relevancy`
   - `faithfulness`, `answer_relevancy`, `hallucination`
   - `all_pass` - True only if ALL 6 metrics pass for that case
5. Print the DataFrame sorted by `all_pass` ascending (failures first).

### Expected Output

A 5-row DataFrame. Cases where `all_pass=False` are the ones to investigate.

### Think About It

Which metric failed most often across your 5 cases? Is it a retriever problem
(contextual_recall / precision) or a generator problem (faithfulness / hallucination)?
That tells you where to focus optimization effort first.

### Stretch

Re-run the same 5 cases with `k=5` instead of `k=3`. Which metrics improve?
Which stay the same? Does higher k help generator metrics or only retriever metrics?

In [ ]:
# =============================================================================
# SOLUTION: Lab 3 - Full 6-Metric Suite on 5 Test Cases
# =============================================================================

six_metrics_lab3 = [
    ContextualRecallMetric(    threshold=0.7, model=haiku_judge),
    ContextualPrecisionMetric( threshold=0.7, model=haiku_judge),
    ContextualRelevancyMetric( threshold=0.7, model=haiku_judge),
    FaithfulnessMetric(        threshold=0.7, model=haiku_judge),
    AnswerRelevancyMetric(     threshold=0.7, model=haiku_judge),
    HallucinationMetric(       threshold=0.3, model=haiku_judge),
]

deepeval_evaluate(test_cases=lab1_cases, metrics=six_metrics_lab3)

# Re-measure per case to build the per-row DataFrame
rows = []
for case in lab1_cases:
    m_recall   = ContextualRecallMetric(    threshold=0.7, model=haiku_judge)
    m_prec     = ContextualPrecisionMetric( threshold=0.7, model=haiku_judge)
    m_rel      = ContextualRelevancyMetric( threshold=0.7, model=haiku_judge)
    m_faith    = FaithfulnessMetric(        threshold=0.7, model=haiku_judge)
    m_ans_rel  = AnswerRelevancyMetric(     threshold=0.7, model=haiku_judge)
    m_halluc   = HallucinationMetric(       threshold=0.3, model=haiku_judge)

    for m in [m_recall, m_prec, m_rel, m_faith, m_ans_rel, m_halluc]:
        m.measure(case)

    # HallucinationMetric passes when score <= threshold (less is better)
    all_pass = (
        m_recall.score  >= 0.7
        and m_prec.score    >= 0.7
        and m_rel.score     >= 0.7
        and m_faith.score   >= 0.7
        and m_ans_rel.score >= 0.7
        and m_halluc.score  <= 0.3
    )
    rows.append({
        "question":             case.input[:55],
        "contextual_recall":    round(m_recall.score,  3),
        "contextual_precision": round(m_prec.score,    3),
        "contextual_relevancy": round(m_rel.score,     3),
        "faithfulness":         round(m_faith.score,   3),
        "answer_relevancy":     round(m_ans_rel.score, 3),
        "hallucination":        round(m_halluc.score,  3),
        "all_pass":             all_pass,
    })

lab3_df = pd.DataFrame(rows).sort_values("all_pass")
print(lab3_df.to_string(index=False))

# Explanation:
# - Sort by all_pass ascending so failures appear first.
# - Common failure pattern: contextual_recall < 0.7 but faithfulness >= 0.7.
#   This means the LLM answered faithfully from what it retrieved, but the retriever
#   missed part of the policy - a regulatory gap invisible to faithfulness alone.
# - HallucinationMetric score of 0 = no contradictions detected (good).
#   Score near 1 = the answer contradicts the retrieved context (bad).

# Section 4: CI-Style Deployment Gate

## From Notebook to CI

DeepEval is Pytest-compatible. Running `deepeval test run test_fraud_rag.py` executes
your evaluation as a standard test suite. This is the hook into Weeks 19-20 (CI/CD):

```
# Makefile / CI step
evaluate:
    deepeval test run tests/test_fraud_rag.py
```

A failing metric (score < threshold) causes the test run to exit with a non-zero
status code, which blocks the GitHub Actions or GitLab CI pipeline.

This lab generates a `test_fraud_rag.py` file in the current directory.
In Week 19 you DVC-version the evaluation dataset and run this file as a CI gate.

## The Deployment Gate Pattern

```python
# test_fraud_rag.py (generated in this lab)
import pytest
from deepeval import assert_test
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, ContextualRecallMetric

@pytest.mark.parametrize("question,expected_output", EVAL_CASES)
def test_fraud_rag(question, expected_output):
    # build test case, run retrieval + generation
    assert_test(test_case, metrics=[
        FaithfulnessMetric(threshold=0.7),
        ContextualRecallMetric(threshold=0.7),
    ])
```

`assert_test` raises `AssertionError` if any metric fails. Pytest collects it.
The `deepeval test run` command also generates a JSON report for the CI dashboard.

In [ ]:
# =============================================================================
# DEMO: Generate test_fraud_rag.py for Week 19 CI Integration
# =============================================================================
# This creates a Pytest file using assert_test (DeepEval's pytest integration).
# Run from terminal: deepeval test run test_fraud_rag.py

DEMO_Q  = "What is the CTR reporting threshold for cash transactions?"
DEMO_GT = (
    "Banks must file a Currency Transaction Report for cash transactions exceeding "
    "$10,000. Multiple same-day transactions by the same person are aggregated. "
    "Structuring to evade the threshold is a federal violation under 31 USC 5324."
)

# Build the test file as a string then write it out
test_file_content = '''\
"""
Fraud RAG Evaluation - CI Deployment Gate
Generated by Week 18 Optional B notebook.
Run: deepeval test run test_fraud_rag.py
"""
import os
import asyncio
import boto3
import pytest
from deepeval import assert_test
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, ContextualRecallMetric, AnswerRelevancyMetric
from deepeval.models.base_model import DeepEvalBaseLLM

MODEL_ID   = "us.anthropic.claude-3-haiku-20240307-v1:0"
AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
KB_ID      = os.environ.get("KNOWLEDGE_BASE_ID", "FARSQGTONR")

_brm = boto3.client("bedrock-runtime",       region_name=AWS_REGION)
_bar = boto3.client("bedrock-agent-runtime", region_name=AWS_REGION)


class HaikuJudge(DeepEvalBaseLLM):
    def load_model(self):
        return _brm

    def generate(self, prompt: str) -> str:
        resp = _brm.converse(
            modelId=MODEL_ID,
            messages=[{"role": "user", "content": [{"text": prompt}]}],
            inferenceConfig={"maxTokens": 1024, "temperature": 0},
        )
        return resp["output"]["message"]["content"][0]["text"]

    async def a_generate(self, prompt: str) -> str:
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(None, self.generate, prompt)

    def get_model_name(self) -> str:
        return MODEL_ID


_judge = HaikuJudge()


def _retrieve(q, k=3):
    resp = _bar.retrieve(
        knowledgeBaseId=KB_ID,
        retrievalQuery={"text": q},
        retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": k}},
    )
    return [i.get("content", {}).get("text", "") for i in resp.get("retrievalResults", [])]


def _answer(q, ctxts):
    ctx    = "\\n".join(ctxts)
    prompt = f"Answer using only the context.\\n\\nContext:\\n{ctx}\\n\\nQuestion: {q}\\nAnswer:"
    resp   = _brm.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 512, "temperature": 0},
    )
    return resp["output"]["message"]["content"][0]["text"]


# Add your (question, ground_truth) pairs here
EVAL_CASES = [
    (
        "''' + repr(DEMO_Q) + '''",
        "''' + repr(DEMO_GT) + '''",
    ),
    # Add more pairs in Lab 4
]


@pytest.mark.parametrize("question,expected_output", EVAL_CASES)
def test_fraud_rag(question, expected_output):
    ctxts = _retrieve(question, k=3)
    ans   = _answer(question, ctxts)
    case  = LLMTestCase(
        input=question,
        actual_output=ans,
        expected_output=expected_output,
        retrieval_context=ctxts,
        context=ctxts,
    )
    assert_test(case, metrics=[
        FaithfulnessMetric(    threshold=0.7, model=_judge),
        ContextualRecallMetric(threshold=0.7, model=_judge),
        AnswerRelevancyMetric( threshold=0.7, model=_judge),
    ])
'''

with open("test_fraud_rag.py", "w") as f:
    f.write(test_file_content)

print("Generated test_fraud_rag.py")
print("Run from terminal: deepeval test run test_fraud_rag.py")
print("Or from notebook:  !deepeval test run test_fraud_rag.py")

## Lab 4: Extend the CI Gate with Your 5 Test Cases (15 min)

### Your Task

Extend `test_fraud_rag.py` with all 5 of your test cases from Lab 1, then run it.

### Steps

1. Read `test_fraud_rag.py` into a string using `open("test_fraud_rag.py").read()`.
2. Extract the (question, ground_truth) pairs from `lab1_cases`:
   - `case.input` is the question
   - `case.expected_output` is the ground truth
3. Rebuild `EVAL_CASES` in the file with all 5 pairs.
4. Write the updated file back.
5. Run it:
   ```python
   !deepeval test run test_fraud_rag.py
   ```
6. Check the exit status. If any metric fails, read the reason string and adjust:
   - If contextual_recall fails consistently, try lowering its threshold to 0.6.
   - Compliance recall on a small KB often needs a slightly relaxed threshold.

### Expected Output

`deepeval test run` output showing pass/fail per test case. The file is now Week 19
ready - you will carry it to the next class.

### Stretch

Add `HallucinationMetric(threshold=0.3)` to the gate. Does it catch anything the
`FaithfulnessMetric` missed? Can you construct a question where hallucination is low
(no contradictions) but faithfulness is also low (vague answer with no supported
statements)?

In [ ]:
# =============================================================================
# SOLUTION: Lab 4 - Extend test_fraud_rag.py with 5 Test Cases
# =============================================================================

# Step 1: Extract pairs from lab1_cases
eval_pairs = [(case.input, case.expected_output) for case in lab1_cases]

# Step 2: Build the EVAL_CASES block as a Python source string
eval_cases_src = "EVAL_CASES = [\n"
for q, gt in eval_pairs:
    eval_cases_src += f"    ({repr(q)},\n     {repr(gt)}),\n"
eval_cases_src += "]\n"

# Step 3: Read the generated test file, replace the EVAL_CASES block, write back
with open("test_fraud_rag.py") as f:
    content = f.read()

# Replace the placeholder EVAL_CASES with our 5 real pairs
import re
content = re.sub(
    r"EVAL_CASES\s*=\s*\[.*?\]\s*\n",
    eval_cases_src,
    content,
    flags=re.DOTALL,
)

with open("test_fraud_rag.py", "w") as f:
    f.write(content)

print("Updated test_fraud_rag.py with 5 test cases.")
print("\nFirst 3 lines of EVAL_CASES:")
for line in eval_cases_src.split("\n")[:4]:
    print(f"  {line}")

# Step 4: Run the CI gate
# !deepeval test run test_fraud_rag.py

# Explanation:
# - repr() safely embeds strings with quotes and special chars in Python source.
# - re.sub with DOTALL replaces the entire multi-line EVAL_CASES block.
# - The # !deepeval line above is commented so students can choose when to run it.
# - If any metric fails: try lowering contextual_recall threshold to 0.6 for small KBs.

# Summary: Deep RAG Evaluation with DeepEval

## Key Takeaways

### Why 6 Metrics, Not 2

- Faithfulness + answer_relevancy (RAGAS) cover the generator. They do NOT cover
  whether the RETRIEVER found everything it should have.
- `ContextualRecallMetric` is the compliance-critical metric: it catches cases where
  "the retriever missed the CTR structuring clause."
- `HallucinationMetric` catches answers that are neither faithful NOR contradictory
  - vague or partially supported responses that slip through faithfulness checks.

### Custom Judge Is Mandatory

- DeepEval's native `BedrockModel` targets Sonnet-class IDs not available in the
  class AWS account.
- The `DeepEvalBaseLLM` custom wrapper is a one-time implementation that works for
  any Bedrock model the account has access to.
- Haiku 3 is the right choice: fast, cheap, and accurate enough for LLM-as-judge
  on a policy-domain evaluation dataset.

### Ground Truth Is the Hard Part

- `ContextualRecallMetric` requires `expected_output` (ground truth).
- Writing ground truth for 5-10 compliance questions takes 30-60 minutes.
- Maintaining it as policies change is a compliance team responsibility.
- This notebook makes that explicit so you understand the real-world cost.

### CI Integration

- `deepeval test run test_fraud_rag.py` is Pytest-compatible.
- Failed metric -> non-zero exit code -> CI block.
- In Week 19, `test_fraud_rag.py` + a DVC-versioned eval dataset = complete CI gate.

## What to Carry Into Week 19

1. `test_fraud_rag.py` (generated in Lab 4)
2. Your (question, ground_truth) pairs as a CSV - DVC will version it
3. The threshold values that passed your compliance review

## Sources

- DeepEval docs: https://deepeval.com/docs
- Custom LLM guide: https://deepeval.com/guides/guides-using-custom-llms
- RAG evaluation guide: https://deepeval.com/guides/guides-rag-evaluation